In [ ]:
"""
SUPERSEDED on 2026-10-06 by notebooks/step01_ibkr_download.ipynb (historical download), notebooks/step02_ibkr_live_stream.ipynb (live bars) and the ingest package.

Kept unchanged (code and saved outputs) as the record of how the raw data was downloaded before that date. It does not
run any more: it imports the old root modules (datetime_utils, local_file_management) that were replaced by the so
package of stock_overflow_workspace. The bugs it had are listed in README.md ("What changed").
"""
None

In [1]:
import pandas as pd
import numpy as np
import time
# import pytz
# import tzlocal
from datetime import datetime, timedelta
import traceback
# IMPORT MARKET DATETIME FUNCTIONS
from datetime_utils import local_tz_name, \
                            local_tz, \
                            ny_tz, \
                            nyse_calendar, \
                            convert_to_ny_ts, \
                            get_date_market_schedule_pdf, \
                            get_date_range_market_schedule_pdf
# FROM LOCAL FILE MANAGEMENT IMPORT
from local_file_management import get_path_file_list, \
                                read_csv_file_from_path, \
                                write_csv_file_to_path, \
                                read_pdf_from_csv_file_path_list, \
                                generate_func_data_date_range, \
                                generate_func_data_date_list

In [2]:
# FUNCTION: GET THE CURRENT TIME IN A SPECIFIC TIMEZONE
def get_datetime_now(timezone_str_in):
    """
    Returns the current datetime according to selected timezone
    """
    # CREATE A DICTIONARY WITH LOCAL AND NEW YORK TIMEZONES
    timezone_dict = {"local": local_tz, "ny": ny_tz}
    # GET THE TIMEZONE OBJECT
    timezone = timezone_dict.get(timezone_str_in)
    # RETURN DATETIME AT TIMEZONE
    return pd.to_datetime(datetime.now(timezone)).replace(microsecond=0)

# # FUNCTION: TESTING
# def get_datetime_now(timezone_str_in):
#     # CREATE A DICTIONARY WITH LOCAL AND NEW YORK TIMEZONES
#     timezone_dict = {"local": local_tz, "ny": ny_tz}
#     # GET THE TIMEZONE OBJECT
#     timezone = timezone_dict.get(timezone_str_in)
#     # RETURN DATETIME AT TIMEZONE
#     return pd.to_datetime("2025-02-14 15:51:12").tz_localize(timezone)

# FUNCTION: GET ADJUSTED MINUTE FLOOR
def get_ts_30sec_minute_floor(timestamp_in):
    """
    Adjusts the given timestamp_in to:
    - The current minute floor if seconds ≥ 30
    - The previous minute floor if seconds < 30
    """
    # IF THE TIMESTAMP SECONDS IS GREATER THAN 30
    if timestamp_in.second >= 30:
        # RETURN THE CURRENT MINUTE FLOOR
        return timestamp_in.replace(second=0, microsecond=0)  # Current minute floor
    # IF THE TIMESTAMP SECONDS IS BELOW 30
    else:
        # RETURN THE PREVIOUS MINUTE FLOOR
        return (timestamp_in - timedelta(minutes=1)).replace(second=0, microsecond=0)  # Previous minute floor
    
# FUNCTION: CREATE IBKR END DATETIME QUERY STRING
def get_ibkr_end_datetime_query_str(datetime_in):
    # CONVERT TIMESTAMP TO IBKR FORMAT
    return (pd.to_datetime(datetime_in) + timedelta(minutes=1)).strftime("%Y%m%d %H:%M:%S")

# COLLECT THE LOCAL TIME
local_datetime_now = get_datetime_now("local")
# COLLECT THE NEW YORK TIME
ny_datetime_now = get_datetime_now("ny")
# GET THE ADJUSTED MINUTE FLOOR
floored_ny_datetime_now = get_ts_30sec_minute_floor(ny_datetime_now)
# COLLECT DATETIME QUERY STRING
end_datetime_str = get_ibkr_end_datetime_query_str(floored_ny_datetime_now)
# CREATE DATAFRAME USING TIMEZONES
display(
    pd.DataFrame({"category": ["LOCAL", "NYSE", "30 Sec Time Floor"],
                    "timezone": [local_tz, ny_tz, ny_tz],
                    "timestamp": [local_datetime_now, ny_datetime_now, floored_ny_datetime_now]})
)
# DISPLAY INFORMATION
print(f"Query String:\t'{end_datetime_str}' (to get data for {floored_ny_datetime_now.strftime('%Y-%m-%d %H:%M:%S')})")

,category,timezone,timestamp
0,LOCAL,Asia/Riyadh,2026-08-14 13:11:06+03:00
1,NYSE,America/New_York,2026-08-14 06:11:06-04:00
2,30 Sec Time Floor,America/New_York,2026-08-14 06:10:00-04:00


Query String:	'20260814 06:11:00' (to get data for 2026-08-14 06:10:00)


In [3]:
# FUNCTION: GET MARKET SCHEDULE FOR TODAY
def get_market_schedule_today(timestamp_in):
    """Get market trading hours for today."""
    # COLLECT THE DATE STRING
    date_str = timestamp_in.strftime("%Y-%m-%d")
    # RETURN THE SCHEDULE DATAFRAME
    return nyse_calendar.schedule(start_date=date_str, end_date=date_str)

# FUNCTION: GET THE NEXT MARKET OPEN DATETIME
def get_next_market_open(timestamp_in):
    """Find the next market opening datetime."""
    # COLLECT THE NEXT OPEN DATE
    next_open_date = nyse_calendar.valid_days(
        start_date=timestamp_in.strftime("%Y-%m-%d"),
        end_date=(timestamp_in + timedelta(days=7)).strftime("%Y-%m-%d")
    )[0]
    # COLLECT THE NEXT OPEN DATE SCHEDULE
    next_schedule_pdf = nyse_calendar.schedule(start_date=next_open_date, end_date=next_open_date)
    # RETURN THE NEXT MARKET OPEN DATE
    return next_schedule_pdf.iloc[0]["market_open"].tz_convert(ny_tz)

# FUNCITON: GET THE ACTIVE RADING HOURS
def get_active_trading_hours(timestamp_in):
    """Return (market_open, market_close) for today, or None if closed."""
    # COLLECT THE ACTIVE TRADING HOURS SCHEDULE
    schedule_pdf = get_market_schedule_today(timestamp_in)
    # IF THE SCHEDULE IS EMPTY
    if schedule_pdf.empty:
        # RETURN NONE VALUES
        return None, None
    # IF THE SCHEDULE CONTAINS DATA
    # RETURN MARKET OPEN & CLOSED
    return (
        schedule_pdf.iloc[0]["market_open"].tz_convert(ny_tz),
        schedule_pdf.iloc[0]["market_close"].tz_convert(ny_tz) - timedelta(minutes=1)  # Adjust close time
    )

# FUNCTION: LOG THE MARKET STATUS
def log_alert(message, alert_in=True):
    """Helper function to log market status messages."""
    # DISPLAY INFORMATION
    print(message) if alert_in==True else None

def get_market_status_dict(datetime_in, alert_in=True):
    """Check if the market is open and return relevant time data."""
    # COLLECT THE DATE STRING
    date_str = datetime_in.strftime("%Y-%m-%d")
    # DISPLAY INFORMATION
    log_alert(f"🕒 New York Time: {datetime_in.strftime('%Y-%m-%d %H:%M:%S')} | 📅 Date: {date_str} | 📆 {datetime_in.strftime('%A')}", alert_in)
    # COLLECT THE MARKET OPEN AND CLOSE VALUES
    market_open, market_close = get_active_trading_hours(datetime_in)
    # IF THE MARKET IS NOT OPEN TODAY
    if market_open is None:
        # DISPLAY INFORMATION
        log_alert("❌ Market is closed today (weekend or holiday).", alert_in)
        # GET THE NEXT OPEN MARKET DATE
        next_open = get_next_market_open(datetime_in)
        # CALCULATE TIME UNTIL OPEN
        time_until_open = next_open - datetime_in
        # DISPLAY INFORMATION
        log_alert(f"📌 Next market opening: {next_open.strftime('%Y-%m-%d %H:%M:%S')} ({time_until_open})", alert_in)
        # RETURN DICTIONARY
        return {"date":date_str,
                "market_open_today":False,
                "market_open": False,
                "seconds_until_open": int(time_until_open.total_seconds()),
                "seconds_open": 0,
                "seconds_until_close": None}
    # DISPLAY INFORMATION
    log_alert(f"✅ Market opens today!", alert_in)
    log_alert(f"🕒 Market Open: \t{market_open.strftime('%Y-%m-%d %H:%M:%S')}", alert_in)
    log_alert(f"🕒 Market Close: \t{market_close.strftime('%Y-%m-%d %H:%M:%S')}\n", alert_in)
    # IF THE MARKET IS CURRENTLY OPEN
    if market_open <= datetime_in <= market_close:
        # CALCULATE THE TIME MARKET HAS BEEN OPEN
        time_open = datetime_in - market_open
        # CALCULATE THE TIME UNTIL MARKET CLOSES
        time_until_close = market_close - datetime_in
        # DISPLAY INFORMATION
        log_alert(f"✅ Market currently open! ⏳ Time until close: {time_until_close}", alert_in)
        log_alert(f"📌 Market closes at: {market_close.strftime('%Y-%m-%d %H:%M:%S')}", alert_in)
        # RETURN DICTIONARY
        return {"date": date_str,
                "market_open_today": True,
                "market_open": True,
                "seconds_until_open": 0,
                "seconds_open": int(time_open.total_seconds()),
                "seconds_until_close": int(time_until_close.total_seconds())}
    # IF THE MARKET HAS NOT OPENED YET
    if datetime_in < market_open:
        # CALCULATE THE TIME UNTIL MARKET OPENS
        time_until_open = market_open - datetime_in
        # DISPLAY INFORMATION
        log_alert(f"❌ Market currently closed! ⏳ Time until open: {time_until_open}", alert_in)
        log_alert(f"📌 Market opens at: {market_open.strftime('%Y-%m-%d %H:%M:%S')}", alert_in)
        # CALCULATE TIME UNTIL CLOSE
        time_until_close = market_close - datetime_in
        # RETURN DICTIONARY
        return {"date": date_str,
                "market_open_today": True,
                "market_open": False,
                "seconds_until_open": int(time_until_open.total_seconds()),
                "seconds_open": 0,
                "seconds_until_close": int(time_until_close.total_seconds())}
    # IF THE MARKET HAS ALREADY OPENED AND CLOSED
    if datetime_in > market_close:
        # CALCULATE THE TIME IT WAS OPEN
        time_open = market_close - market_open
        # COLLECT THE NEXT OPEN DATETIME
        next_open = get_next_market_open(datetime_in + timedelta(days=1))
        # CALCULATE TIME UNTIL NEXT OPEN
        time_until_open = next_open - datetime_in
        # DISPLAY INFORMATION
        log_alert(f"❌ Market is currently closed. ⏳ Time until next open: {time_until_open}", alert_in)
        log_alert(f"📌 Next market opens at: {next_open.strftime('%Y-%m-%d %H:%M:%S')}", alert_in)
        # RETURN DICTIONARY
        return {"date": date_str,
                "market_open_today": True,
                "market_open": False,
                "seconds_until_open": int(time_until_open.total_seconds()),
                "seconds_open": int(time_open.total_seconds()),
                "seconds_until_close": 0}

# RUN THE MARKET STATUS CHECK
market_status_dict = get_market_status_dict(ny_datetime_now, alert_in=True)

🕒 New York Time: 2026-08-14 06:11:06 | 📅 Date: 2026-08-14 | 📆 Friday
✅ Market opens today!
🕒 Market Open: 	2026-08-14 09:30:00
🕒 Market Close: 	2026-08-14 15:59:00

❌ Market currently closed! ⏳ Time until open: 0 days 03:18:54
📌 Market opens at: 2026-08-14 09:30:00


In [4]:
from ibapi.client import *
from ibapi.wrapper import *
import threading
from threading import Event
from ibapi.ticktype import TickTypeEnum

# DEFINE THE CLASS TEST APP 
class TestApp(EClient, EWrapper):
    # INSTANTIATE THE CLASS BY DEFINING AN init METHOD
    def __init__(self):
        # INITIALIZE THE EClient
        EClient.__init__(self, self)
        # DEFINE DICTIONARY TO HOLD ORDER IDS AND TIMESTAMPS
        self.orderId_timestamp_dict = {}
        # DEFINE A LIST TO STORE THE HISTORICAL DATA
        self.historical_data_list = []
        # EVENT TO SIGNAL DATA READINESS
        self.data_ready_event = threading.Event()
        # DEFINE DATA STREAMING
        self.data_streaming = False
        # DEFINE A VARIABLE TO GET THE LATEST TIMESTAMP
        self.latest_data_timestamp = None
        # DEFINE A LIST TO STORE LIVE DATA
        self.live_data_list = []

    # METHOD: WRAPPER OBJECT TO RECEIVE ORDER IDs (THIS WILL RECIEVE THE EWRAPPER AND RETURN THE NEXT ORDER ID TO PREVENT DUPLICATE VALUES) 
    def nextValidId(self, orderId):
        # STORE ORDER ID WITH TIMESTAMP
        self.orderId_timestamp_dict[orderId] = datetime.now().strftime("%Y-%m-%d %H:%M:%S") 
        # RETURN THE SELF ORDER ID
        self.orderId = orderId
    
    # METHOD: NEXT ID WILL RETAIN THE OBJECT (TRACK REQUESTS)
    def nextId(self):
        # INCREMENT THE NEWLY MADE ORDER ID BY 1 AND RETURN THE ORDER ID VALUE
        self.orderId += 1
        # RECORD THE TIMESTAMP OF THE NEW ORDER ID
        self.orderId_timestamp_dict[self.orderId] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        # RETURN THE SELF ORDER ID
        return self.orderId

    # METHOD: GET THE ORDER ID TIMESTAMP
    def get_order_timestamp(self, orderId):
        # RETURN THE ORDER TIMESTAMP FOR THE ORDER ID
        return self.orderId_timestamp_dict.get(orderId, "Timestamp not found")
    
    # METHOD: RETURN THE CURRENT EMPOCH TIME OBSERVED IN TWS
    def currentTime(self, time):
        # DISPLAY INFORMATION
        print(time)

    # METHOD: DISPLAY ERROR WITH PROCESSING A REQUEST (any error generated by the TWS while processing your request such as invalid order formats, market data issues, and more)
    def error(self, reqId, errorTime, errorCode, errorString, advancedOrderReject="", advancedOrderRejectJson=""):
        # DISPLAY INFORMATION
        print(f"reqId: {reqId}, errorTime: {errorTime}, errorCode: {errorCode}, errorString: {errorString}, "
                f"orderReject: {advancedOrderReject}, advancedOrderRejectJson: {advancedOrderRejectJson}")
        # CHECK FOR SUCCESSFUL CONNECTION
        if errorCode == 2106:
            # DISPLAY INFORMATION
            print("✅ Connection to TWS established!")

    # METHOD: CONTAINS ALL THE CONTRACT INFORMATION IN A DICTIONARY STRUCTURE
    def contractDetails(self, reqId, contractDetails):
        # COLLECT THE CONTRACT DETAILS
        attrs = vars(contractDetails)
        # DISPLAY INFORMATION
        print("\n".join(f"{name}: {value}" for name,value in attrs.items()))

    # METHOD: INDICATES THERE IS NO DATA REMAINING FOR A REQUEST THAT MAY RETURN SEVERAL RESPONSES
    def contractDetailsEnd(self, reqId):
        # DISPLAY INFORMATION
        print("End of contract details")
        # DISCONNECT APP
        self.disconnect()

    # METHOD: COLLECT THE OLDEST DATA FOR A STOCK
    def headTimestamp(self, reqId, headTimeStamp):
        # DISPLAY INFORMATION
        print("Oldest Available Unix Time:", headTimeStamp)
        # CANCEL THE HEAD TIMESTAMP REQUEST
        self.cancelHeadTimeStamp(reqId)

    # METHOD: HANDLE RETURNING VALUES RELATED TO PRICE VALUES
    def tickPrice(self, reqId, tickType, price, attrib):
        # DISPLAY INFORMATION
        print(f"reqId: {reqId}, tickType: {TickTypeEnum.toStr(tickType)}, price: {price}, attrib: {attrib}")

    # METHOD: THE SIZES HERE RELATE TO THE PRICES RETURNED IN OUR tickPrice FUNCTION
    def tickSize(self, reqId, tickType, size):
        # DISPLAY INFORMATION
        print(f"reqId: {reqId}, tickType: {TickTypeEnum.toStr(tickType)}, size: {size}")
        
    # METHOD: REQUEST HISTORICAL DATA
    def historicalData(self, reqId, bar):
        # CONVERT THE UNIX TIMESTAMP TO DATETIME OBJECT
        data_timestamp_object = datetime.fromtimestamp(int(bar.date))
        # COLLECT THE DATETIME IN NEW YORK
        ny_data_timestamp = pd.to_datetime(data_timestamp_object.astimezone(ny_tz))
        # COLLECT THE DATETIME NOW
        ny_datetime_now = pd.to_datetime(datetime.now().astimezone(ny_tz))
        # IF LIVE DATA IS BEING STREAMED
        if self.data_streaming:
            # STORE THE HISTORICAL DATA IN HISTORICAL DATA LIST
            self.live_data_list.append((ny_data_timestamp, bar.open, bar.high, bar.low, bar.close, bar.volume, ny_datetime_now))
            # SET THE LATEST DATA TIMESTAMP
            self.latest_data_timestamp = ny_data_timestamp
        # IF HISTORICAL DATA IS BEING QUERIED
        else:
            # STORE THE HISTORICAL DATA IN HISTORICAL DATA LIST
            self.historical_data_list.append((ny_data_timestamp, bar.open, bar.high, bar.low, bar.close, bar.volume, ny_datetime_now))
        # DISPLAY INFORMATION
        # print(f"reqId: {reqId}\t{bar}")
    
    # METHOD: USED TO IDENTIFY WHEN WE ARE DONE REQUESTING DATA
    def historicalDataEnd(self, reqId, start, end):
        # DISPLAY INFORMATION
        print(f"Historical Data Ended for reqId: {reqId}. Started at {start}, ending at {end}")
        # SIGNAL THAT THE DATA COLLECTION IS COMPLETE
        self.data_ready_event.set()
        # CANCEL HISTORICAL DATA REQUEST USING ID
        self.cancelHistoricalData(reqId)

    # METHOD: CONNECT IF NOT ALREADY CONNECTED
    def connect_if_needed(self):
        # IF THE APP IS NOT CONNECTED
        if not self.isConnected():
            # DISPLAY INFORMATION
            print("⚠️ Not connected, attempting to connect...")
            # CONNECT TO THE HOST IP ADDRESS (LOCALHOST), PORT, CLIENT ID
            # self.connect("127.0.0.1", 7497, 0) # TWS Paper Trasing
            # self.connect("127.0.0.1", 7496, 0)  # TWS Live Trading
            self.connect("127.0.0.1", 4001, 0)  # IB Gateway Live Trading
            # START THREAD
            threading.Thread(target=self.run, daemon=True).start()  # Start the app in a separate thread
        # IF THE APP IS CONNECTED
        else:
            # DISPLAY INFORMATION
            print("✅ Already connected to TWS!")
    
    # METHOD: DISCONNECT CLEANLY WHEN NEEDED
    def disconnect_app(self):
        # CHECK IF THE APP IS CONNECTED
        if self.isConnected():
            # DISPLAY INFORMATION
            print("✅ Disconnecting from TWS...")
            # DISCONNECT FROM APP
            self.disconnect()

# ATTEMPT CHECK APP INSTANTIATION
try:
    # IF THE APP EXISTS AND IT IS CONNECTED
    if app.isConnected():
        # DISPLAY INFORMATION
        print(f"✅ Already connected to TWS!")
    # IF THE APP IS NOT CONNECTED
    else:
        # CONNECT TO TWS
        app.connect_if_needed()
# IF THE APP DOES NOT EXIST
except NameError as e:
    # INSTANTIATE THE TEST APPLICATION OBJECT
    app = TestApp()
    # CALL METHOD TO CONNECT TO TWS
    app.connect_if_needed()

⚠️ Not connected, attempting to connect...


reqId: -1, errorTime: 1786702266819, errorCode: 2104, errorString: Market data farm connection is OK:usfarm, orderReject: , advancedOrderRejectJson: 
reqId: -1, errorTime: 1786702266831, errorCode: 2106, errorString: HMDS data farm connection is OK:ushmds, orderReject: , advancedOrderRejectJson: 
✅ Connection to TWS established!
reqId: -1, errorTime: 1786702266832, errorCode: 2158, errorString: Sec-def data farm connection is OK:secdefil, orderReject: , advancedOrderRejectJson: 


In [5]:
app.isConnected() # CHECK IF THE APP IS CONNECTED
# app.disconnect_app() # DISCONNECT THE APP
# app.connect_if_needed() # CALL METHOD TO CONNECT TO TWS (IF CONNECTION IS NEEDED)

True

In [6]:
def AAPL_contract_NASDAQ():
    contract = Contract()
    contract.symbol = "AAPL"
    contract.secType = "STK"
    contract.currency = "USD"
    contract.exchange = "BATS"
    return contract

# mycontract = AAPL_contract_NASDAQ() # Requested market data requires additional subscription for API. See link in 'Market Data Connections' 

def AAPL_contract_NYSE():
    contract = Contract()
    contract.symbol = "AAPL"
    contract.secType = "STK"
    contract.currency = "USD"
    contract.exchange = "NYSE"  # Change to NYSE
    return contract

# mycontract = AAPL_contract_NYSE() #  Requested market data requires additional subscription for API. See link in 'Market Data Connections' 

# FUNCTION: DEFINE SP500 CONTRACT
def SPY_contract_BATS():
    # INSTANTIATE CONTRACT
    contract = Contract()
    contract.symbol = "SPY"  # SPDR S&P 500 ETF
    contract.secType = "STK"  # Stock
    contract.currency = "USD"
    contract.exchange = "BATS"  # Request data from BATS Exchange
    return contract

# mycontract = SPY_contract_BATS() # requires additional subscription for API. See link in 'Market Data Connections'

def AAPL_contract():
    contract = Contract()
    contract.symbol = "AAPL"
    contract.secType = "STK"
    contract.exchange = "SMART"  # Use "SMART" for best exchange routing
    contract.currency = "USD"
    return contract

# mycontract = AAPL_contract() # WORKS FOR SNAPSHOT REAL TIME

# FUNCTION: DEFINE SP 500 CONTRACT
def create_contract(symbol_str_in, sec_type_str_in, currency_str_in, exchange_str_in, primary_exchange_str_in):
    # INSTANTIATE A CONTRACT
    new_contract = Contract()
    # DEFINE CONTRACT ATTRIBUTES
    new_contract.symbol = symbol_str_in
    new_contract.secType = sec_type_str_in
    new_contract.currency = currency_str_in
    new_contract.exchange = exchange_str_in
    new_contract.primaryExchange = primary_exchange_str_in
    # RETURN CONTRACT
    return new_contract

# # CALL FUNCTION TO GET CONTRACT
mycontract = create_contract("SPY", "STK", "USD", "SMART", "ARCA")

# FUNCTION: PREVIEW CONTRACT
def preview_contract_details(contract_in):
    # COLLECT CONTRACT VALUES
    print(f"|======== Contract ========|")
    print(f"Symbol:\t\t\t{contract_in.symbol}")
    print(f"Security Type:\t\t{contract_in.secType}")
    print(f"Currency:\t\t{contract_in.currency}")
    print(f"Exhange:\t\t{contract_in.exchange}")
    print(f"Primary Exchange:\t{contract_in.primaryExchange}")
    print(f"|==========================|")

# PREVIEW CONTRACT DETAILS
preview_contract_details(mycontract)

|======== Contract ========|
Symbol:			SPY
Security Type:		STK
Currency:		USD
Exhange:		SMART
Primary Exchange:	ARCA
|==========================|


In [7]:
# SET REAL TIME DATA TYPE
app.reqMarketDataType(1) # LIVE DATA
# app.reqMarketDataType(2) # FROZEN DATA
# app.reqMarketDataType(3) # DELAYED DATA
# app.reqMarketDataType(4) # DELAYED FROZEN DATA

In [8]:
# FUNCTION TO QUERY HISTORICAL DATA AND WAIT FOR RESULTS
def query_historical_1min_ohlcv(contract_in, end_datetime_in, seconds_duration_in):
    # IF APP IS NOT CONNECTED EXIT FUNCTION
    if app.isConnected() == False:
        # DISPLAY INFORMATION
        print("App is not connected!")
        # RETURN NOTHING
        return None
    # ATTEMPT TO QUERY DATA
    try:
        # COLLECT THE CURRENT TIMESTAMP
        start_timestamp = datetime.now()
        # DEFINE THE QUERY END DATETIME STRING
        end_datetime_str = get_ibkr_end_datetime_query_str(end_datetime_in) + " US/Eastern"
        # DEFINE THE QUERY SECONDS 
        seconds_duration_str = f"{int(seconds_duration_in)} S"
        # RESET HISTORICAL DATA LIST
        app.historical_data_list = []
        # RESET THE DATA READY EVENT
        app.data_ready_event.clear()
        # SEND THE HISTORICAL DATA REQUEST
        """
        reqId:              Unique identifier for the historical data request.
        contract:           Contract object specifying the asset (symbol, secType, exchange, etc.).
        endDateTime:        The end date and time of the data request (formatted as "YYYYMMDD-HH:MM:SS").
        durationStr:        The duration of the historical data request (e.g., "1 D" for 1 day, "1 W" for 1 week).
        barSizeSetting:     The granularity of the bars (e.g., "1 min", "5 mins", "1 hour").
        whatToShow:         The type of data to return (e.g., "TRADES", "MIDPOINT", "BID", "ASK").
        useRTH:             1 to use regular trading hours, 0 to include extended hours data.
        formatDate:         1 to format dates as strings, 2 to return as timestamps.
        keepUpToDate:       True to receive real-time updates after the historical request.
        """
        # COLLECT THE NEXT ORDER ID
        reqId = app.nextId()
        # app.reqHistoricalData(app.nextId(), SP500_contract, "20200115 09:00:00 US/Eastern", "1 W", "1 min", "TRADES", 1, 2, False, []) # STATIC QUERY
        app.reqHistoricalData(reqId = reqId,
                              contract = contract_in,
                              endDateTime = end_datetime_str,
                              durationStr = seconds_duration_str,
                              barSizeSetting = "1 min",
                              whatToShow = "TRADES",
                              useRTH = 1,
                              formatDate = 2,
                              keepUpToDate = False,
                              chartOptions = []) # DYNAMIC QUERY
        # SET AN APPROPRIATE TIMEOUT DURATION
        timeout_seconds = 10
        # WAIT FOR THE DATA TO BE COLLECTED WITH A TIMEOUT
        if not app.data_ready_event.wait(timeout=timeout_seconds):
            # RAISE TIMEOUT ERROR
            raise TimeoutError(f"Request timed out after {timeout_seconds} seconds.")
        # GET THE EXECUTION END TIMESTAMP
        end_timestamp = datetime.now()
        # GET THE TIME DIFFERENCE
        execution_seconds_duration = (end_timestamp - start_timestamp).total_seconds()
        # DISPLAY INFORMATION
        print(f"\nQuery Returned {len(app.historical_data_list)} Results! (in {execution_seconds_duration} seconds) (reqId: {reqId})\n")
    # COLLECT TIMEOUT ERROR
    except TimeoutError as te:
        # DISPLAY INFORMATION
        print(f"TimeoutError: {te}")
    # COLLECT EXCEPTION ERROR
    except Exception as e:
        # DISPLAY INFORMATION
        print(f"An error occurred: {e}")
        # PRINT THE TRACEBACK
        traceback.print_exc()
    # ENSURE THE EVENT IS SET TO PREVENT HANGING
    finally:
        # SET EVENT
        app.data_ready_event.set()

# CALL FUNCTION TO QUERY HISTORICAL DATA
# query_historical_1min_ohlcv(mycontract, "20250205 12:59:00", 12600) # WORKS
# query_historical_1min_ohlcv(mycontract, "20250205 12:59:00", 12600) # WORKS
# query_historical_1min_ohlcv(mycontract, "20250207 16:00:00", 300) # (5 minutes) WORKS
# query_historical_1min_ohlcv(mycontract, "20250207 16:00:00", 300) # (5 minutes) WORKS
# query_historical_1min_ohlcv(mycontract, "20250326 16:00:00", 23400)
# query_historical_1min_ohlcv(mycontract, "20250326 16:00:00", 300)
# query_historical_1min_ohlcv(mycontract, end_datetime_query_str, 60)
# query_historical_1min_ohlcv(mycontract, ny_datetime_now, 60)

In [9]:
# app.historical_data_list = []
hist_data_pdf = pd.DataFrame(app.historical_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_at"]).sort_values(by=["timestamp"])
print(f"Min Timestamp:\t{hist_data_pdf.timestamp.min()}")
print(f"Min Timestamp:\t{hist_data_pdf.timestamp.max()}")
display(hist_data_pdf)

Min Timestamp:	nan
Min Timestamp:	nan


,timestamp,open,high,low,close,volume,created_at


In [22]:
# FUNCTION: GENERATE AND SAVE DATA FOR A DATE
def get_date_ohlcv_pdf(date_str_in):
    # DEFINE COLUMN STRING LIST
    col_str_list = ["timestamp", "open", "high", "low", "close", "volume", "created_ts"]
    # COLLECT THE DATE MARKET SCHEDULE
    date_market_schedule_pdf = get_date_market_schedule_pdf(date_str_in)
    # IF THE DATE MARKET SCHEDULE IS EMPTY RETURN AN EMPTY DATAFRAME
    if date_market_schedule_pdf.empty:
        # RETURN AN EMPTY DATAFRAME
        return pd.DataFrame(columns=col_str_list)
    # COLLECT THE MARKET OPEN AND CLOSE TIME
    market_open_time = date_market_schedule_pdf.iloc[0]["market_open_ts"]
    market_close_time = date_market_schedule_pdf.iloc[0]["market_close_ts"]
    # CALCULATE THE NUMBER OF SECONDS BETWEEN THE MARKET OPEN AND CLOSE TIME
    seconds_duration = (market_close_time - market_open_time).total_seconds() + 60
    # QUERY THE DATA FROM IBKR
    query_historical_1min_ohlcv(mycontract, market_close_time, seconds_duration)
    # COLLECT THE HISTORICAL DATA
    date_ohlcv_pdf = pd.DataFrame(app.historical_data_list, columns=col_str_list).sort_values(by=["timestamp"])
    # SET CREATED TS TO NULL
    date_ohlcv_pdf["created_ts"] = np.nan
    # RETURN THE DATAFRAME
    return date_ohlcv_pdf

# DEFINE SAMPLE DATE
# sample_date_str = "20090105"
# sample_date_str = "2006-11-22"
# sample_date_str = "2005-05-26"
sample_date_str = "2005-07-01"
# sample_date_str = "20250326"
# sample_date_str = "20250822"
# sample_date_str = "20250929"
# sample_date_str = "20260717"
# CALL FUNCTION TO GET DATA FOR A DATE
date_ohlcv_pdf = get_date_ohlcv_pdf(sample_date_str)
# PREVIEW DATAFRAME
date_ohlcv_pdf

Historical Data Ended for reqId: 922. Started at 20050701 09:30:00 US/Eastern, ending at 20050701 16:00:00 US/Eastern

Query Returned 390 Results! (in 0.915805 seconds) (reqId: 922)

reqId: 922, errorTime: 1786705236405, errorCode: 366, errorString: No historical data query found for ticker id:922, orderReject: , advancedOrderRejectJson: 


,timestamp,open,high,low,close,volume,created_ts
0,2005-07-01 09:30:00-04:00,119.45,119.46,119.41,119.44,453200,NaN
1,2005-07-01 09:31:00-04:00,119.44,119.48,119.44,119.45,222100,NaN
2,2005-07-01 09:32:00-04:00,119.46,119.48,119.44,119.45,326000,NaN
3,2005-07-01 09:33:00-04:00,119.46,119.51,119.45,119.49,196700,NaN
4,2005-07-01 09:34:00-04:00,119.49,119.52,119.47,119.47,168200,NaN
...,...,...,...,...,...,...,...
385,2005-07-01 15:55:00-04:00,119.44,119.48,119.41,119.48,181500,NaN
386,2005-07-01 15:56:00-04:00,119.48,119.50,119.45,119.50,133200,NaN
387,2005-07-01 15:57:00-04:00,119.50,119.53,119.50,119.51,141200,NaN
388,2005-07-01 15:58:00-04:00,119.51,119.53,119.45,119.51,103800,NaN


In [13]:
# DEFINE SAVE DATA FILE PATH
OHLCV_DATA_FILE_PATH_STR = "//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/"

In [24]:
# DEFINE DATETIME 1
# date1_str = "20250326"
# date1_str = datetime.now().strftime("%Y%m%d")
# date1_str = "20260101"
# DEFINE DATETIME 2
# date2_str = "20250330"
# date2_str = "20260811"
# date2_str = datetime.now().strftime("%Y%m%d")
# date2_str = "20260101"
# date1_str, date2_str = "20080101", "20090222" # DONE
# date1_str, date2_str = "20070101", "20080101" # DONE
# date1_str, date2_str = "20061122", "20070101" # DONE
# date1_str, date2_str = "20050701", "20060101" # need to execute
date1_str, date2_str = "20050101", "20050101" # need to execute
# CALL FUNCTION TO GENERATE FUNCTION DATA FOR DATE RANGE
generate_func_data_date_range(get_date_ohlcv_pdf, date1_str, date2_str, OHLCV_DATA_FILE_PATH_STR, "ohlcv_data", "I")

❌ No market schedule found for date range: 20050101 to 20050101


""


In [61]:
# # DEFINE MISSING DATE LIST
# missing_date_list = ['2025-03-26', '2025-03-27', '2025-03-28', '2025-03-31', '2025-06-12', '2025-08-25', '2025-09-30', '2025-11-04', '2025-12-10', '2026-02-24', '2026-05-06', '2026-06-11', '2026-07-20', '2026-08-12']
# missing_date_list = ['2005-03-15', '2005-04-20', '2005-05-25', '2005-06-30', '2005-08-08', '2005-10-18', '2005-11-21', '2006-02-08', '2006-03-16', '2006-04-21', '2006-06-30', '2006-08-08', '2006-09-13', '2006-11-21', '2006-12-29', '2007-02-09', '2007-03-16', '2007-04-23', '2007-05-29', '2007-07-02', '2007-08-08', '2007-09-14', '2007-10-19', '2007-12-31', '2008-02-07', '2008-03-17', '2008-04-21', '2008-05-28', '2008-07-01', '2008-08-07', '2008-10-16', '2008-12-29', '2009-02-04', '2026-08-13']
# missing_date_list = ['2007-07-02', '2008-04-21']
# missing_date_list = ['2005-09-12', '2005-12-29', '2006-05-26', '2006-10-18', '2007-11-26', '2008-09-12', '2008-11-21', '2009-07-27', '2013-12-23'] # PARTIAL DATE LIST
# missing_date_list = ['2007-07-02'] +  ['2009-07-27', '2013-12-23'] # MISSING DAY + PARTIAL DATE LIST
# missing_date_list = ['2005-03-15', '2005-04-20', '2005-05-25', '2005-06-30', '2005-08-08', '2005-10-18', '2005-11-21', '2006-02-08', '2006-03-16', '2006-04-21', '2006-06-30', '2006-08-08', '2006-09-13', '2006-11-21', '2006-12-29', '2007-02-09', '2007-03-16', '2007-04-23', '2007-05-29', '2007-07-02', '2007-08-08', '2007-09-14', '2007-10-19', '2007-12-31', '2008-02-07', '2008-03-17', '2008-04-21', '2008-05-28', '2008-07-01', '2008-08-07', '2008-10-16', '2008-12-29', '2009-02-02', '2009-02-03', '2009-02-04', '2009-02-05', '2009-02-06', '2009-02-09', '2009-02-10', '2009-02-11', '2009-02-12', '2009-02-13', '2009-02-17', '2009-02-18', '2009-02-19', '2009-02-20', '2025-03-26', '2025-03-27', '2025-03-28', '2025-03-31', '2025-06-12', '2025-08-25', '2025-09-30', '2025-11-04', '2025-12-10', '2026-08-14']
# missing_date_list = ['2007-07-02', '2008-04-21', '2025-06-12']
# missing_date_list = ['2005-09-12', '2005-12-29', '2006-05-26', '2006-10-18', '2007-11-26', '2008-09-12', '2008-11-21', '2009-07-27', '2013-12-23', '2026-08-14']
# missing_date_list = ['2007-07-02', '2008-04-21']
# missing_date_list =  ['2009-07-27', '2013-12-23']
missing_date_list = []
# # PREVIEW LIST
# missing_date_list

In [62]:
# CALL FUNCTION TO GENERATE FUNCTION DATA FOR DATE LIST
generate_func_data_date_list(get_date_ohlcv_pdf, missing_date_list, OHLCV_DATA_FILE_PATH_STR, "ohlcv_data", "I")

""


In [58]:
# # GET DATA FILE PATH LIST
# data_file_path_list = get_path_file_list(OHLCV_DATA_FILE_PATH_STR)
# # PREVIEW LIST
# data_file_path_list

In [59]:
# # CALL FUNCTION TO READ PDF FROM PATH LIST
# raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(data_file_path_list)
# # PREVIEW DATAFRAME
# raw_ohlcv_pdf

In [14]:
# query_historical_1min_ohlcv(mycontract, "2000-01-07 15:59:00", 60*390) # TOO FAR BACK
# query_historical_1min_ohlcv(mycontract, "2000-01-10 15:59:00", 60*390) # FIND OUT HOW FAR BACK I CAN GO (WORK STARTS HERE)
# query_historical_1min_ohlcv(mycontract, "2005-01-01 15:59:00", 60*390) # WORKS
# query_historical_1min_ohlcv(mycontract, "2005-01-07 15:59:00", 60*390) # WORKS
# query_historical_1min_ohlcv(mycontract, "2009-01-08 15:59:00", 60*390) # TOO FAR BACK
# query_historical_1min_ohlcv(mycontract, "2009-02-24 15:59:00", 23400) # EARLIEST TIME THAT I AM ABLE TO QUERY FOR THE WHOLE DAY (AS OF 2025-02-19)
# query_historical_1min_ohlcv(mycontract, "2010-01-08 15:59:00", "1 W") # EARLIEST THAT WORKS
# query_historical_1min_ohlcv(mycontract, "2010-01-08 15:59:00", "1 M") # EARLIEST THAT WORKS  (DOES WORK BUT TAKES A WHILE
# query_historical_1min_ohlcv(mycontract, "2009-03-01 23:59:59", "1 Y") # READING 1 YEAR OF DATA (DOES WORK BUT TAKES A WHILE)
# query_historical_1min_ohlcv(mycontract, "2010-11-23 16:00:00", 390*60) # THIS RETURNS 391 ROWS UP TO 16:00:00 INSTEAD OF 15:59:00
# query_historical_1min_ohlcv(mycontract, "2010-11-23 15:59:59", 390*60) # THIS RETURNS 391 ROWS UP TO 16:00:00 INSTEAD OF 15:59:00
# query_historical_1min_ohlcv(mycontract, "2010-11-23 23:59:59", 390*60) # THIS RETURNS DATA UP TO 16:14:00 INSTEAD OF 15:59:00
# query_historical_1min_ohlcv(mycontract, "2013-12-23 23:59:59", "1 D") # THIS DATE DOES NOT CONTAIN ALL THE DATA (352/390 ROWS)
# query_historical_1min_ohlcv(mycontract, "2009-07-27 23:59:59", "1 D") # THIS DATE DOES NOT CONTAIN ALL THE DATA (285/390 ROWS)
# query_historical_1min_ohlcv(mycontract, "2020-03-18 23:59:59", "1 D") # THIS DATE CONTAINS DATA BUT FROM 12:57 TO 13:10 OHLC ARE ALL THE SAME VALUES AND V IS 0
# query_historical_1min_ohlcv(mycontract, "2009-12-08 15:59:00", "23400 S") # FIGURE OUT LATEST DATE THAT WORKS
# query_historical_1min_ohlcv(mycontract, "2025-02-18 23:59:59", 390*60)
# query_historical_1min_ohlcv(mycontract, "2025-03-26 16:00:00", 23400) # COLLECT DATA FOR DATE

In [15]:
# app.historical_data_list = []
hist_data_pdf = pd.DataFrame(app.historical_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_ts"]).sort_values(by=["timestamp"])
print(f"Min Timestamp:\t{hist_data_pdf.timestamp.min()}")
print(f"Min Timestamp:\t{hist_data_pdf.timestamp.max()}")
display(hist_data_pdf)

Min Timestamp:	2026-07-17 09:30:00-04:00
Min Timestamp:	2026-07-17 15:59:00-04:00


,timestamp,open,high,low,close,volume,created_ts
0,2026-07-17 09:30:00-04:00,742.05,742.52,741.80,742.35,821961,2026-08-12 19:36:24.602833-04:00
1,2026-07-17 09:31:00-04:00,742.36,742.62,742.10,742.51,420477,2026-08-12 19:36:24.603838-04:00
2,2026-07-17 09:32:00-04:00,742.53,742.56,741.84,741.85,406041,2026-08-12 19:36:24.603838-04:00
3,2026-07-17 09:33:00-04:00,741.87,742.00,741.28,741.32,221188,2026-08-12 19:36:24.603838-04:00
4,2026-07-17 09:34:00-04:00,741.33,741.45,741.01,741.03,199812,2026-08-12 19:36:24.603838-04:00
...,...,...,...,...,...,...,...
385,2026-07-17 15:55:00-04:00,744.00,744.07,743.67,744.03,491697,2026-08-12 19:36:24.674679-04:00
386,2026-07-17 15:56:00-04:00,744.03,744.40,744.03,744.31,239057,2026-08-12 19:36:24.674679-04:00
387,2026-07-17 15:57:00-04:00,744.31,744.33,743.98,744.07,319733,2026-08-12 19:36:24.674679-04:00
388,2026-07-17 15:58:00-04:00,744.06,744.10,743.84,744.01,448539,2026-08-12 19:36:24.674679-04:00


In [16]:
from sqlalchemy import create_engine, MetaData, Table, text, Column, Integer, String, Float, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

# DEFINE DATABASE CONNECTION SETTINGS
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "mysecretpassword"
DB_HOST = "localhost"
DB_PORT = "5432"
# DEFINE A DATABASE URI
DB_URI = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
# CREATE A SQLALCHEMY ENGINE
engine = create_engine(DB_URI)

# FUNCTION: CONNECT TO DATABASE AND CHECK CONNECTION
def connect_to_db(alert_in=True):
    # TRY TO CONNECT
    try:
        # USE ENGINE TO CONNECT TO DATABASE
        with engine.connect() as conn:
            # DISPLAY INFORMATION
            print("✅ Timescale DB is: Connected") if alert_in==True else None
            # RETURN TRUE
            return True
    # CATCH EXCEPTION
    except Exception as e:
        # DISPLAY INFORMATION
        print(f"❌ Timescale DB is: Disconnected ({e})") if alert_in==True else None
        # RETURN FALSE
        return False

# CALL FUNCTION TO CONNECT TO DT
# connected = connect_to_db()

In [17]:
# FUNCTION: INSERT MULTIPLE ROWS INTO A TABLE
def insert_multiple_rows(table_name, data):
    """
    Dynamically insert multiple rows into a given table by name.

    :param table_name: Name of the table to insert data into (string).
    :param data: A list of dictionaries containing row data.
                 Example: [{"symbol": "AAPL", "price": 150.5, "volume": 1000, "timestamp": datetime.now()}, ...]
    """
    # CREATE A SESSION
    Session = sessionmaker(bind=engine)
    # CREATE A SESSION
    session = Session()
    # COLLECT METADATA
    metadata = MetaData()
    # TRY TO INSERT DATA
    try:
        # REFLECT EXISTING TABLES IN THE DATABASE
        metadata.reflect(engine)
        # ENSURE THAT TABLE EXISTS
        if table_name not in metadata.tables:
            # RAISE ERROR
            raise ValueError(f"⚠️ Table '{table_name}' does not exist in the database.")
        # GET THE TABLE OBJECT DYNAMICALLY
        table = metadata.tables[table_name]
        # INSERT MULTIPLE ROWS INTO THE DATA
        session.execute(table.insert(), data)
        # COMMIT THE CHANGES
        session.commit()
        # DISPLAY INFORMATION
        print(f"✅ {len(data)} rows inserted successfully into '{table_name}'.")
    # CATCH EXCEPTION
    except Exception as e:
        # ROLLBACK THE SESSION
        session.rollback()
        # DISPLAY INFORMATION
        print(f"❌ Error inserting data into '{table_name}': {e}")
    # FINALLY
    finally:
        # CLOSE THE SESSION
        session.close()

# # EXAMPLE DATA
# stock_data = [
#     {"symbol": "AAPL", "price": 150.5, "volume": 1000, "timestamp": datetime.now()},
#     {"symbol": "GOOG", "price": 2800.0, "volume": 500, "timestamp": datetime.now()},
#     {"symbol": "MSFT", "price": 299.7, "volume": 1200, "timestamp": datetime.now()},
#     {"symbol": "XD", "price": 70, "volume": 500, "timestamp": datetime.now()}
# ]
# INSERT MULTIPLE ROWS 
# insert_multiple_rows('stock_prices', stock_data)

In [18]:
# # FUNCTION: VERIFY WEEKLY HISTORICAL DATA
# def upload_data_to_DB(list_in):
#     # LIST TO HOLD VALUES
#     stock_data_list = []
#     # ITERATE OVER DATA IN LIST
#     for timestamp, open, high, low, close, volume, created_at in list_in:
#         # CREATE A ROW DICTIONARY
#         row_dict = {"timestamp": timestamp,
#                     "open": open,
#                     "high": high,
#                     "low": low,
#                     "close": close,
#                     "volume": volume,
#                     "created_at": created_at}
#         # ADD DICTIONARY TO DATA LIST
#         stock_data_list.append(row_dict)
#     # INSERT MULTIPLE ROWS 
#     # insert_multiple_rows('stock_prices', stock_data_list)
#     return stock_data_list

# CALL FUNCTION TO INSERT DATA LIST
# test_out = upload_data_to_DB(app.historical_data_list[0:100])
# PREVIEW
# test_out

In [19]:
# EVENT TO CONTROL STREAMING
stop_streaming = threading.Event()
# TRACK WHEN STREAMING STARTS
streaming_start_time = None
# TRACK THE ACTIVE THREAD
streaming_thread = None

# FUNCTION: QUERY LIVE 1 MINUTE DATA
def query_live_1min_ohlcv(datetime_in):
    """
    Queries 1-minute OHLCV data for mycontract every minute asynchronously, starting at the next full minute.
    Stops when the queried timestamp reaches market close or when manually stopped.
    Tracks elapsed time and displays when stopped.
    Only 30 seconds after the minute has elapsed will the data for the previous minute be generated
    """
    # COLLECT THE ACTIVE TRADING HOURS
    market_open, market_close = get_active_trading_hours(datetime_in)
    # IF STREAM IS HAPPENING AFTER MARKET TRADING HOURS
    if datetime_in > market_close + timedelta(seconds=30):
        # DISPLAY INFORMATION
        print(f"⚠️ Market has already closed! Streaming not necessary")
        # EXIT FUNCTION
        return
    # COLLECT GLOBAL STREAM START TIME, STOP STREAM, AND STREAM THREAD
    global streaming_start_time, stop_streaming, streaming_thread
    # ENSURE PREVIOUS STREAMING IS FULLY STOPPED
    stop_live_1min_ohlcv()
    # RESET THE EVENT TO ALLOW A NEW THREAD TO RUN
    stop_streaming = threading.Event()
    # ENSURE THAT IBKR IS CONNECTED
    app.connect_if_needed()
    # CAPTURE THE START TIME
    streaming_start_time = time.time()
    # SET STREAMING STATE TO TRUE
    app.data_streaming = True
    # NESTED FUNCTION: TO FETCH DATA
    def fetch_data():
        # COLLECT THE STERAMING START TIME
        global streaming_start_time
        # CALCULATE WHEN TO START EXECUTION
        query_timestamp = datetime_in.replace(second=0)
        query_execution_timestamp = datetime_in.replace(second=30)
        execution_seconds_diff = int((query_execution_timestamp - datetime_in).total_seconds())
        # DEFINE EXECUTION SECONDS INTERVAL
        execution_seconds_interval = 60
        # DISPLAY INFORMATION
        print(f"Attempting to execute query at:\t\t{datetime_in.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Query will collect data for:\t\t{query_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Query should be executed at:\t\t{query_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Default execution time interval:\t{execution_seconds_interval} seconds")
        print(f"\tFirst query needs to be executed in {execution_seconds_diff} seconds")
        # IF THE EXECUTION SECONDS DIFFERENCE IS POSITIVE
        if execution_seconds_diff > 0:
            # DISPLAY INFORMATION
            print(f"⏳ Waiting {execution_seconds_diff} seconds to start at next interval {query_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
            # WAIT FOR EXECUTION TIME DIFFERENCE
            time.sleep(execution_seconds_diff)
        # IF THE EXECUTION SECONDS TIME DIFFERENCE IS NEGATIVE
        else:
            # SUBTRACT EXECUTION TIME DIFFERENCE FROM EXECUTION SCONDS INTERVAL
            execution_seconds_interval = execution_seconds_interval + execution_seconds_diff
            # DISPLAY INFORMATION
            print(f"Next execution time interval will be: {execution_seconds_interval} seconds")
        # DISPLAY INFORMATION
        print(f"🕒 Will stream until data reaches {market_close.strftime('%Y-%m-%d %H:%M:%S')}")
        # COUNT EXECUTIONS
        counter = 0
        # RUN UNTIL STOPPED
        while not stop_streaming.is_set():
            # GET THE QUERY END DATETIME STRING
            query_end_str = get_ibkr_end_datetime_query_str(query_timestamp)
            # DEFINE THE SECONDS DURATION
            duration_str = "35 S"
            # DISPLAY INFORMATION
            print(f"\n📡 Querying 1 minute data for {query_timestamp.strftime('%Y-%m-%d %H:%M:%S')} (execution: {counter})")
            # REQUEST THE RELEVANT 1 MINUTE DATA
            app.reqHistoricalData(reqId = app.nextId(),
                                  contract = mycontract,
                                  endDateTime = query_end_str,
                                  durationStr = duration_str,
                                  barSizeSetting = "1 min",
                                  whatToShow = "TRADES",
                                  useRTH = 1,
                                  formatDate = 2,
                                  keepUpToDate = False,
                                  chartOptions = [])
            # IF THE LIVE DATA STARTED EXECUTING
            if counter > 0:
                # SET EXECUTION SECONDS INTERVAL TO 60
                execution_seconds_interval = 60
            # DISPLAY INFORMATION
            print(f"⏳ Next execution in {execution_seconds_interval} seconds")
            # WAIT FOR NEXT FULL EXECUTION
            # time.sleep(execution_seconds_interval)
            time.sleep(5) # TESTING
            # INCREASE QUERY TIMESTAMP
            query_timestamp += timedelta(minutes=1)
            # INCREASE COUNTER
            counter += 1
            # IF THE QUERY TIMESTAMP EXCEEDS THE MARKET CLOSE
            if query_timestamp > market_close:
                # DISPLAY INFORMATION
                print(f"\n🛑 Reached market close time: {market_close}. Stopping data streaming.")
                # SET THE STOP STREAMING VARIABLE
                stop_streaming.set()
                # SET DATA STREAMING TO FALSE
                app.data_streaming = False
                # CALCULATE ELAPSED TIME
                elapsed_time = time.time() - streaming_start_time
                # DISPLAY INFORMATION
                print(f"⛔ Live data streaming stopped. Total runtime: {elapsed_time:.2f} seconds.")
                # SET START STERAMING THREAD TO NONE
                streaming_start_time = None
                # SET STERAMING THREAD TO NONE
                streaming_thread = None
                # BREAK LOOP
                break
    # START A NEW THREAD SAFELY
    streaming_thread = threading.Thread(target=fetch_data, daemon=True)
    streaming_thread.start()

# FUNCTION: STOP LIVE 1 MINUTE OHLCV DATA
def stop_live_1min_ohlcv():
    """Stops the live data streaming process and ensures the thread is fully terminated before restarting."""
    # COLLECT GLOBAL STREAM START TIME, STOP STREAM, AND STREAM THREAD
    global streaming_start_time, streaming_thread, stop_streaming
    # IF LIVE DATA IS NOT BEING STREAMED
    if app.data_streaming == False:
        print(f"⚠️ live OHLCV data is not being streamed.")
        return
    # SET THE STOP STREAMING THREAD EVENT
    stop_streaming.set()
    # SET DATA STREAMING TO FALSE
    app.data_streaming = False
    # IF THE THREAD IS CURRENTLY ACTIVE
    if streaming_thread and streaming_thread.is_alive():
        # DISPLAY INFORMATION
        print("⏳ Waiting for the current streaming thread to stop...")
        # WAIT FOR THE THREAD TO FULLY TERMINATE
        streaming_thread.join()  # Wait for the thread to fully terminate
    # IF START TIME EXISTS
    if streaming_start_time:
        # CALCULATE ELAPSED TIME
        elapsed_time = time.time() - streaming_start_time
        # DISPLAY INFORMATION
        print(f"\n⛔ Live data streaming stopped. Total runtime: {elapsed_time:.2f} seconds.")
    else:
        # DISPLAY INFORMATION
        print(f"\n⛔ Live data streaming stopped.")
    # SET START STERAMING THREAD TO NONE
    streaming_start_time = None
    # SET STERAMING THREAD TO NONE
    streaming_thread = None

# # ATTEMPT TO QUERY LIVE DELAYED DATA
# ny_datetime_now = get_datetime_now("ny")
# # CALL FUNCTION TO QUERY LIVE 1 MINUTE INTERVAL DATA
# query_live_1min_ohlcv(ny_datetime_now)

In [20]:
# # EVENT TO CONTROL STREAMING
# stop_streaming = threading.Event()
# # ATTEMPT TO QUERY LIVE DELAYED DATA
# ny_datetime_now = get_datetime_now("ny")
# # CALL FUNCTION TO QUERY LIVE
# query_live_1min_ohlcv(ny_datetime_now)

In [21]:
# CALL FUNCTION TO STOP LIVE STREAMING DATA
stop_live_1min_ohlcv()

⚠️ live OHLCV data is not being streamed.


In [22]:
# CHECK IF DATA STREAMING IS ACTIVE
app.data_streaming

False

In [23]:
# ITERATE OVER ALL THREADS
for thread in threading.enumerate():
    # DISPLAY INFORMATION
    print(thread.name)

MainThread
IOPub
Heartbeat
Control
Shell channel
IPythonHistorySavingThread
Thread-1
Thread-3
Thread-4 (run)


In [24]:
# app.live_data_list = []

In [25]:
live_data_pdf = pd.DataFrame(app.live_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_at"]).sort_values(by=["timestamp"])
# live_data_pdf["latency"] = (live_data_pdf.created_at - live_data_pdf.timestamp)
display(live_data_pdf)

,timestamp,open,high,low,close,volume,created_at


In [26]:
# FUNCTION: IDENTIFY COLLECTED AND REQUIRED DATA FOR TODAY
def get_required_data_pdf(datetime_in):
    # COLLECT THE ACTIVE TRADING HOURS
    market_open, market_close = get_active_trading_hours(datetime_in)
    # CREATE DATAFRAME OF EXPECTED MINUTES
    expected_timestamps_pdf = pd.DataFrame({"timestamp":pd.date_range(market_open, market_close, freq="1min")})
    # COMBINE HISTORICAL AND LIVE DATA LISTS
    accumulated_data_list = app.historical_data_list + app.live_data_list
    # CONVERT LISTS TO DATAFRAME
    accumulated_data_pdf = pd.DataFrame(accumulated_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_at"]).sort_values(by=["timestamp"])
    # MERGE EXPECTED TIMESTAMP LIST WITH TOTAL DATA LIST
    complete_data_pdf = expected_timestamps_pdf.merge(accumulated_data_pdf, on=["timestamp"], how="left")
    # COLLECT MISSING DATA
    missing_data_pdf = complete_data_pdf[complete_data_pdf.open.isna()].copy()
    # CREATE A LABEL FOR LIVE OR HISTORICAL DATA (SUBTRACT 30 SCEONDS TO REFLECT 30 SECOND LATENCY)
    missing_data_pdf["label"] = np.where((missing_data_pdf.timestamp < datetime_in - timedelta(seconds=30)), "hist", "live")
    # AGGREGATE TIMESTAMP BY LABEL
    timestamp_agg_pdf = missing_data_pdf.groupby("label")["timestamp"].agg(["min", "max", "count"]).reset_index().rename(columns={"min":"min_ts",
                                                                                                                                  "max":"max_ts",
                                                                                                                                  "count":"ts_count"})
    # CALCULATE THE NUMBER OF SECONDS BETWEEN THE MINIMUM AND MAXIMUM TIMESTAMPS
    timestamp_agg_pdf["seconds_duration"] = (timestamp_agg_pdf.max_ts - timestamp_agg_pdf.min_ts).dt.total_seconds().astype(int)
    # PREVIEW DATAFRAME
    return timestamp_agg_pdf

# CALL FUNCTION TO IDENTIFY COLLECTED AND REQUIRED DATA
required_data_pdf = get_required_data_pdf(ny_datetime_now)
# PREVIEW DATAFRAME
required_data_pdf

,label,min_ts,max_ts,ts_count,seconds_duration
0,hist,2026-08-12 09:30:00-04:00,2026-08-12 15:59:00-04:00,390,23340


In [27]:
# EVENT TO CONTROL STREAMING
stop_streaming = threading.Event()

# FUNCTION: DETERMINE STREAMING START AND/OR HISTORICAL REQUEST
def determine_streaming_time(timestamp_in, mycontract_in):
    """Determine when to start streaming data based on market hours."""
    # COLLECT THE MARKET STATUS
    market_status_dict = get_market_status_dict(timestamp_in, alert_in=False)  # Using optimized function
    # COLLECT THE ACTIVE TRADING HOURS
    market_open, market_close = get_active_trading_hours(timestamp_in)
    # IF THE MARKET IS CLOSED
    if market_open is None or market_close is None:
        # DISPLAY INFORMATION
        print("❌ Market is closed today. Streaming not required.")
        # EXIT FUNCTION
        return
    # DISPLAY INFORMATION
    print("✅ Determining Data Stream")
    # CALCULATE REAL TIME RELATEED VALUES
    total_market_open_seconds = (market_close - market_open).total_seconds()
    # COLLECT THE MARKET OPEN METADATA INFORMATION
    seconds_open = market_status_dict["seconds_open"]
    seconds_until_open = market_status_dict["seconds_until_open"]
    seconds_until_close = market_status_dict["seconds_until_close"]

    # CALL FUNCTION TO IDENTIFY COLLECTED AND REQUIRED DATA
    required_data_pdf = get_required_data_pdf(ny_datetime_now)
    # IF THERE IS NO REQUIRED DATA
    if required_data_pdf.empty:
        # DISPLAY INFORMATION
        print("✅ All Data Already Collected")
        # EXIT FUNCTION
        return
    # COLLECT REQUIRED HISTORICAL DATA
    required_hist_data_pdf = required_data_pdf[required_data_pdf.label == "hist"]
    # COLLECT REQUIED LIVE DATA
    required_live_data_pdf = required_data_pdf[required_data_pdf.label == "live"]
    # DEFINE BOOL TO COLLECT HISTORICAL DATA
    collect_hist_data_bool = required_hist_data_pdf.empty == False
    # DEFINE BOOL TO COLLECT LIVE DATA
    collect_live_data_bool = required_live_data_pdf.empty == False

    # IF HISTORICAL DATA NEEDS TO BE COLLECTED
    if collect_hist_data_bool:
        # COLLCET COUNT
        count = required_hist_data_pdf.iloc[0]["ts_count"]
        # DISPLAY INFORMATION
        print(f"⚠️ Need to collect: {count} rows of historical data")
    # IF LIVE DATA NEEDS TO BE COLLECTED
    if collect_live_data_bool:
        # COLLCET COUNT
        count = required_live_data_pdf.iloc[0]["ts_count"]
        # DISPLAY INFORMATION
        print(f"⚠️ Need to collect: {count} rows of live data")

    # CASE 1: THE MARKET HAS NOT OPENNED YET
    if (seconds_open == 0):
        # COLLECT THE START STREAMING TIME
        market_open_str = market_open.strftime('%Y-%m-%d %H:%M:%S')
        # DISPLAY INFORMATION
        print(f"⏳ Market opens in {seconds_until_open} seconds at {market_open_str}.")
        print(f"Waiting for market to open ...")
        # Uncomment if you want the script to wait until the market opens
        time.sleep(seconds_until_open)
        # time.sleep(15) # TESTING
        # DISPLAY INFORMATION
        print("\n✅ Market is open! Starting data stream...")
        print(f"Streaming started at: {market_open} for {seconds_until_close} seconds.")
        # START DATA STREAM
        query_live_1min_ohlcv(market_open)
        
    # CASE 2: MARKET IS ALREADY OPEN
    elif (seconds_open < total_market_open_seconds):
        # DISPLAY INFORMATION
        print(f"✅ Market has been open for {seconds_open} seconds")
        # COLLECT START TIME
        start_time = time.time()
        # IF MARKET HAS BEEN OPEN FOR MORE THAN 90 SECONDS (1 HISTORICAL ROW HAS ALREADY PASSED)
        if (seconds_open >= 90):
            # GET THE EXPECTEDE LAST TIMESTAMP
            expected_last_timestamp = get_ts_30sec_minute_floor(timestamp_in)
            # DISPLAY INFORMATION
            print(f"🕒 Expected last collected timestamp: {expected_last_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
            # IF THE HISTORICAL DATA LIST HAS ITEMS (HISTORICAL DATA HAS BEEN COLLECTED)
            if app.historical_data_list:
                # DISPLAY INFORMATION
                print(f"⚠️ Historical data does exists!")
                # COLLECT LAST HISTORICAL DATA TIMESTAMP
                latest_historical_timestamp = app.historical_data_list[-1][0]
                # IF THE EXPECTED LATEST HISTORICAL TIMESTAMP IS LESS THAN THE EXPECTED TIMESTAMP (REFILL HISTORICAL DATA)
                if (latest_historical_timestamp < expected_last_timestamp):
                    # DISPLAY INFORMATION
                    print(f"⚠️ Historical data is not up to date!")
                    print(f"\nCollecting historical data for past {seconds_open} seconds.")
                    # QUERY HISTORICAL DATA
                    query_historical_1min_ohlcv(mycontract_in, expected_last_timestamp, f"{seconds_open} S")
            # IF THE HISTORICAL DATA DOES NOT EXIST (HISTORICAL DATA HAS NOT BEEN COLLCETED)
            else:
                # DISPLAY INFORMATION
                print(f"⚠️ Historical data does not exist!")
                print(f"\nCollecting historical data for past {seconds_open} seconds.")
                # QUERY HISTORICAL DATA
                query_historical_1min_ohlcv(mycontract_in, expected_last_timestamp, f"{seconds_open - 60} S")
                # DEFINE COUNTER
                counter = 0
                # DISPLAY INFORMATION
                print(f"⏳ Waiting for historical data to be collected... (for 6 seconds)")
                # IF HISTORICAL DATA EXISTS
                while not app.historical_data_list:
                    # WAIT FOR 1 SECOND
                    time.sleep(1)
                    # INCREASE COUNTER
                    counter += 1
                    # IF COUNTER IS GREATER THAN 10
                    if counter > 5:
                        # DISPLAY INFORMATION
                        print(f"⚠️ Historical data has not been collected after 10 seconds. Exiting...")
                        # EXIT FUNCTION
                        return
                # COLLECT LAST HISTORICAL DATA TIMESTAMP
                latest_historical_timestamp = app.historical_data_list[-1][0]
            # DISPLAY INFORMATION
            print(f"✅ Historical data collected until: {latest_historical_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        # COLLECT END TIME
        end_time = time.time()
        # COLLECT SECONDS DURATION
        seconds_duration = (end_time - start_time)
        # INCREASE THE DATETIME IN BY THE AMOUNT OF TIME IT TOOK
        current_timestamp = timestamp_in + timedelta(seconds = seconds_duration)
        # DEFINE THE NEXT QUERY EXECUTION TIMESTAMP
        next_execution_timestamp = latest_historical_timestamp + timedelta(seconds=90)
        # CALCULATE TIME DIFFERENCE BETWEEN CURRENT TIMESTAMP AND BUFFERED LATEST HISTORICAL TIMESTAMP
        execution_seconds_diff = (next_execution_timestamp - current_timestamp).total_seconds()
        # CALCULATE SECOND DIFFERENCE BETWEEN CURRENT TIME AND LAST TIMESTAMP
        print(f"\nCurrent timestamp:\t\t{current_timestamp}")
        print(f"Latest historical timestamp:\t{latest_historical_timestamp}")
        print(f"Next execution timestamp at:\t{next_execution_timestamp}\n")
        # IF THE EXECUTION SECONDS DIFFERENCE IS POSITIVE
        if (execution_seconds_diff > 0):
            # DISPLAY INFORMATION
            print(f"⏳ Waiting {execution_seconds_diff} seconds to start stream at next interval {next_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
            # WAIT FOR EXECUTION SECONDS DIFFERENCE
            time.sleep(execution_seconds_diff)
            # time.sleep(2) # TESTING
        # DISPLAY INFORMATION
        print(f"\n📡 Streaming started at: {next_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')} for {seconds_until_close} seconds.")
        # START DATA STREAM
        query_live_1min_ohlcv(next_execution_timestamp)

    # CASE 3: MARKET WAS OPEN BUT IS NOW CLOSED
    else:
        # DISPLAY INFORMATION
        print(f"⚠️ Market closed. Streaming not required.")
        print(f"Historical data available for {seconds_open} seconds.")
        # IF THE HISTORICAL DATA HAS ALREADY BEEN COLLECTED
        if (collect_hist_data_bool == False):
            # DISPLAY INFORMATION
            print("✅ Historical Data Has Already Been Collected")
            # EXIT FUNCTION
            return
        # QUERY HISTORICAL DATA FOR ENTIRE MARKET DAY (60 SECONDS BUFFER)
        query_historical_1min_ohlcv(mycontract_in, market_close, f"{seconds_open + 60} S")

# GET THE CURRENT NEW YORK DATE TIME
ny_datetime_now = get_datetime_now("ny")
ny_datetime_15mins_ago = ny_datetime_now - timedelta(minutes=15)
print(f"\n<=== TIMESTAMP: {ny_datetime_now} =====>",)
print(f"<=== TIMESTAMP 15 MINS AGO: {ny_datetime_15mins_ago} =====>\n",)
# EXECUTE FUNCTION TO COLLECT STREAM DATA
# determine_streaming_time(ny_datetime_15mins_ago, mycontract) # DELAYED DATA
# determine_streaming_time(ny_datetime_now, mycontract) # LIVE DATA


<=== TIMESTAMP: 2026-08-12 19:38:31-04:00 =====>
<=== TIMESTAMP 15 MINS AGO: 2026-08-12 19:23:31-04:00 =====>



In [28]:
app.isConnected()
# app.connect_if_needed()

True

In [29]:
# CALL FUNCTION TO STOP LIVE STREAMING DATA
stop_live_1min_ohlcv()

⚠️ live OHLCV data is not being streamed.


In [30]:
# app.live_data_list = []
# app.historical_data_list = []

In [31]:
app.data_streaming

False

In [32]:
live_data_pdf = pd.DataFrame(app.live_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_at"]).sort_values(by=["timestamp"])
# live_data_pdf["latency"] = (live_data_pdf.created_at - live_data_pdf.timestamp)
display(live_data_pdf)

,timestamp,open,high,low,close,volume,created_at


In [33]:
hist_data_pdf = pd.DataFrame(app.historical_data_list, columns=["timestamp", "open", "high", "low", "close", "volume", "created_at"]).sort_values(by=["timestamp"])
display(hist_data_pdf)

,timestamp,open,high,low,close,volume,created_at
0,2026-07-17 09:30:00-04:00,742.05,742.52,741.80,742.35,821961,2026-08-12 19:36:24.602833-04:00
1,2026-07-17 09:31:00-04:00,742.36,742.62,742.10,742.51,420477,2026-08-12 19:36:24.603838-04:00
2,2026-07-17 09:32:00-04:00,742.53,742.56,741.84,741.85,406041,2026-08-12 19:36:24.603838-04:00
3,2026-07-17 09:33:00-04:00,741.87,742.00,741.28,741.32,221188,2026-08-12 19:36:24.603838-04:00
4,2026-07-17 09:34:00-04:00,741.33,741.45,741.01,741.03,199812,2026-08-12 19:36:24.603838-04:00
...,...,...,...,...,...,...,...
385,2026-07-17 15:55:00-04:00,744.00,744.07,743.67,744.03,491697,2026-08-12 19:36:24.674679-04:00
386,2026-07-17 15:56:00-04:00,744.03,744.40,744.03,744.31,239057,2026-08-12 19:36:24.674679-04:00
387,2026-07-17 15:57:00-04:00,744.31,744.33,743.98,744.07,319733,2026-08-12 19:36:24.674679-04:00
388,2026-07-17 15:58:00-04:00,744.06,744.10,743.84,744.01,448539,2026-08-12 19:36:24.674679-04:00


In [34]:
# COLLECT MINIMUM AND MAXIMUM TIMESTAMPS
h_min_timestamp, h_max_timestamp = hist_data_pdf.timestamp.min(), hist_data_pdf.timestamp.max()
l_min_timestmap, l_max_timestamp = live_data_pdf.timestamp.min(), live_data_pdf.timestamp.max()
# COLLECT DATA STATS LIST
data_list = [("hist",h_min_timestamp, h_max_timestamp), ("live",l_min_timestmap, l_max_timestamp)]
# DISPLAY INFORMATION
print(f"Queried At: {ny_datetime_now.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"For Date: {market_status_dict['date']}")
# CREATE A DATAFRAME TO HOLD STATS
pd.DataFrame(data_list, columns=["category","min_timestamp","max_timestamp"])

Queried At: 2026-08-12 19:38:31
For Date: 2026-08-12


,category,min_timestamp,max_timestamp
0,hist,2026-07-17 09:30:00-04:00,2026-07-17 15:59:00-04:00
1,live,NaT,NaT


In [35]:
# # EVENT TO CONTROL STREAMING
# stop_streaming = threading.Event()
# # TRACK WHEN STREAMING STARTS
# streaming_start_time = None

# # FUNCTION: QUERY LIVE 1 MINUTE DATA
# def query_live_1min_ohlcv(datetime_in):
#     """
#     Queries 1-minute OHLCV data for mycontract every minute asynchronously, starting at the next full minute.
#     Stops when the queried timestamp reaches market close or when manually stopped.
#     Tracks elapsed time and displays when stopped.
#     Only 30 seconds after the minute has elapsed will the data for the previous minute be generated
#     """
#     # ENSURE THAT IBKR IS CONNECTED
#     app.connect_if_needed()
#     # COLLECT THE GLOBAL STREAMING START TIME
#     global streaming_start_time
#     # CAPTURE THE START TIME
#     streaming_start_time = time.time()
#     # SET STREAMING STATE TO TRUE
#     app.data_streaming = True
#     # GET THE MARKET CLOSE TIME
#     market_close = get_last_market_close(datetime_in)
#     # NESTED FUNCTION: TO FETCH DATA
#     def fetch_data():
#         # DEFINE THE QUERY MINUTE TIMESTAMP
#         query_timestamp = datetime_in.replace(second=0)
#         # DEFINE QUERY EXECUTION TIME TIMESTAMP
#         query_execution_timestamp = datetime_in.replace(second=30)
#         # CALCULATE THE DIFFERENCE BETWEEN THE DATETIME IN AND THE QUERY EXECUTION TIMESTAMP
#         execution_seconds_diff = int((query_execution_timestamp - datetime_in).total_seconds())
#         # DEFINE AN EXECUTION TIME INTERVAL
#         execution_time_interval = 60
#         # DISPLAY INFORMATION
#         print(f"Attempting to execute query at:\t\t{datetime_in.strftime('%Y-%m-%d %H:%M:%S')}")
#         print(f"Query will collect data for:\t\t{query_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
#         print(f"Query should be executed at:\t\t{query_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
#         print(f"Default execution time interval:\t{execution_time_interval} seconds")
#         print(f"\tFirst query needs to be executed in {execution_seconds_diff} seconds")
#         # IF THE SECOND EXECUTION SECONDS DIFFERENCE IS POSITIVE
#         if execution_seconds_diff > 0:
#             # DISPLAY INFORMATION
#             print(f"⏳ Waiting {execution_seconds_diff} seconds to start at next interval {query_execution_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
#             # WAIT FOR EXECUTION SECONDS DIFFERENCE
#             time.sleep(execution_seconds_diff)
#         # IF EXECUTION SECOND SECONDS DIFFERENCE IS NEGATIVE (EXECUTION HAPPENING LATE)
#         else:
#             # SUBTRACT EXECUTION SECONDS FROM TIME INTERVAL
#             execution_time_interval = execution_time_interval + execution_seconds_diff
#             # DISPLAY INFORMATION
#             print(f"Next execution time interval will be: {execution_time_interval} seconds")
#         # DISPLAY INFORMATION
#         print(f"🕒 Will stream until data reaches {market_close.strftime('%Y-%m-%d %H:%M:%S')}")
#         # DEFINE COUNTER
#         counter = 0
#         # RUN UNTIL STOPPED
#         while not stop_streaming.is_set():
#             # CONVERT TIMESTAMP TO IBKR FORMAT
#             query_end_str = get_ibkr_end_datetime_query_str(query_timestamp)
#             # DEFINE SECONDS DURATION
#             duration_str = "35 S" # PAST 35 SECONDS (WORKS)
#             # duration_str = "31 S" # PAST 31 SECONDS
#             # QUERY HISTORICAL DATA
#             print(f"\n📡 Querying 1 minute data for {query_timestamp.strftime('%Y-%m-%d %H:%M:%S')} (execution: {counter})")
#             # REQUEST THE LATEST 1-MIN OHLCV DATA
#             app.reqHistoricalData(reqId = app.nextId(),
#                                     contract = mycontract,
#                                     endDateTime = query_end_str,
#                                     durationStr = duration_str,
#                                     barSizeSetting = "1 min",
#                                     whatToShow = "TRADES",
#                                     useRTH = 1,
#                                     formatDate = 2,
#                                     keepUpToDate = False,
#                                     chartOptions = [])
#             # IF THERE ARE ITEMS IN THE LIVE DATA (IT ALREADY STARTED EXECUTING)
#             if counter > 0:
#                 # SET EXECUTION INTERVAL TO 60
#                 execution_time_interval = 60
#             # DISPLAY INFORMATION
#             print(f"⏳ Next execution in {execution_time_interval} seconds")
#             # WAIT FOR THE NEXT FULL MINUTE
#             time.sleep(execution_time_interval)
#             # time.sleep(5) # TESTING
#             # INCREASE THE QUERY TIMESTAMP BY 1 MINUTE
#             query_timestamp += timedelta(minutes=1)
#             # INCREASE EXECUTION COUNTER BY 1
#             counter += 1
#             # IF THE DATETIME EXCEEDS THE MARKET CLOSE
#             if query_timestamp > market_close:
#                 # DISPLAY INFORMATION
#                 print(f"🛑 Reached market close time: {market_close}. Stopping data streaming.")
#                 # STOP FUNCTION
#                 stop_live_1min_ohlcv()
#                 # BREAK WHILE LOOP
#                 break
#     # START THE DATA FETCH PROCESS IN A SEPARATE THREAD
#     thread = threading.Thread(target=fetch_data, daemon=True)
#     # START THE THREAD
#     thread.start()

# # FUNCTION: STOP LIVE 1 MINUTE OHLCV DATA
# def stop_live_1min_ohlcv():
#     """Stops the live data streaming process and displays the elapsed time."""
#     # IF LIVE DATA IS NOT BEING STREAMED
#     if app.data_streaming == False:
#         # DISPLAY INFORMATION
#         print(f"⚠️ live OHLCV data is not being streamed.")
#         # EXIT FUNCTION
#         return
#     # COLLECT THE GLOBAL STREAMING TIME
#     global streaming_start_time
#     # STOP STREAMING
#     stop_streaming.set()
#     # SET DATA STREAMING TO FALSE WHEN STREAMING STOPS
#     app.data_streaming = False
#     # IF THE STREAMING STARTED
#     if streaming_start_time:
#         # CALCULATE ELAPSED TIME
#         elapsed_time = time.time() - streaming_start_time
#         # DISPLAY INFORMATION
#         print(f"⛔ Live data streaming stopped. Total runtime: {elapsed_time:.2f} seconds.")
#     # IF THERE IS NO START TIME
#     else:
#         # DISPLAY INFORMATION
#         print("⛔ Live data streaming stopped.")
#     # RESET START TIME
#     streaming_start_time = None

# # ATTEMPT TO QUERY LIVE DELAYED DATA
# # ny_datetime_15mins_ago = pd.to_datetime(datetime.now().astimezone(ny_tz)) - timedelta(minutes=15)
# ny_datetime_now = get_datetime_now("ny")
# # CALL FUNCTION TO QUERY LIVE 1 MINUTE INTERVAL DATA
# # query_live_1min_ohlcv(ny_datetime_15mins_ago)
# query_live_1min_ohlcv(ny_datetime_now)

In [36]:
# # FUNCTION: REQUEST LIVE MARKET DATA
# def query_ibkr_snapshot_live_market_data(contract):
#     # IF THE APP IS NOT CONNECTED
#     if not app.isConnected():
#         # DISPLAY INFORMATION
#         print("App is not connected!")
#         # EXIT FUNCTION
#         return
#     # ATTEMPT TO REQUEST LIVE MARKET DATA
#     try:
#         # COLLECT THE NEXT ORDER ID
#         reqId = app.nextId()
#         """ 
#         reqId:                  Unique identifier for the market data request.
#         contract:               Contract object specifying the asset (symbol, secType, exchange, etc.).
#         genericTicks:           Comma-separated string for additional market data (e.g., "233" for implied volatility).
#         snapshot:               If True, request a one-time snapshot of market data, else stream live updates.
#         regulatorySnapshot:     If True, request market data with regulatory requirements (usually False).
#         marketDataOptions:      List of additional options for the market data request (usually empty).
#         """
#         # app.reqMktData(reqId, contract, "", False, False, []) # THIS WILL KEEP PULLING DATA (IDK HOW TO STOP IT)
#         app.reqMktData(reqId = reqId,
#                        contract = contract,
#                        genericTickList = "", 
#                        snapshot = True,
#                        regulatorySnapshot = False,
#                        mktDataOptions = []) # THIS WILL ONLY COLLECT A SNAPSHOT OF THE DATA
#         # DISPLAY INFORMATION
#         print(f"Live market data request sent for {contract.symbol} on {contract.exchange} (reqId: {reqId})")
#     # CATCH EXCEPTOIN
#     except Exception as e:
#         # DISPLAY INFORMATION
#         print(f"Error requesting live market data: {e}")

# # CALL FUNCTION TO REQUEST LIVE MARKET DATA
# query_ibkr_snapshot_live_market_data(mycontract)

In [37]:
# # FUNCTION: COLLECT TEST LIVE DATA COLLECTION
# def test_live_data():
#     # GET THE TIMESTAMP NOW
#     datetime_now = datetime.now().replace(microsecond=0)
#     # INCREASE DATETIME NOW BY 1 MINUTE
#     datetime_1min_later = datetime_now + timedelta(minutes=1)
#     # CREATE DATETIME WITH 0 SECONDS
#     start_datetime = datetime_1min_later.replace(second=0)
#     # DEFINE END DATE TIME
#     end_datetime = start_datetime + timedelta(minutes=5)
#     # CREATE A DATE TIME RANGE
#     min_range_list = list(pd.date_range(start_datetime, end_datetime, freq="1min"))
#     # DISPLAY INFORMATION
#     print(f"Minutes: {min_range_list}")
#     # DISPLAY INFORMATION
#     print(f"Start Time: {datetime_now}\n")
#     # WHILE THE CURRENT TIME IS LESS THAN THE END DATE TIME
#     while len(min_range_list):
#         # DISPLAY INFORMATION
#         print(f"Current Time: {datetime.now().replace(microsecond=0)} waiting...")
#         # IF THE DATETIME NOW IS IN THE DATE RANGE LIST
#         if datetime.now().replace(microsecond=0) in min_range_list:
#             # DISPLAY INFORMATION
#             print("⏰ Time to collect data!")
#             # CALL FUNCTION TO COLLECT DATA
#             print(f"Collecting Data At: {datetime.now().replace(microsecond=0)}")
#             # CALL FUNCTION TO REQUEST LIVE MARKET DATA
#             query_snapshot_live_market_data(mycontract)
#             # REMOVE MINUTE FROM THE LIST
#             min_range_list.pop(0)
#             # DISPLAY INFORMATION
#             print(f"New Min Range List: {min_range_list}")
#         # SLEEP FOR 1 SECOND
#         time.sleep(1)

# # CALL FUNCTION TO TEST LIVE DATA
# # test_live_data()

In [38]:
# # Request real-time 5-second bars from ES futures contract
# app.reqRealTimeBars(
#     reqId = app.nextId(),
#     contract = mycontract,
#     barSize = 5,  # IBKR only supports 5-second real-time bars
#     whatToShow = "TRADES",
#     useRTH = 0,  # Use all trading hours
#     realTimeBarsOptions = []
# )